<div style="background:linear-gradient(90deg,#0b3d91,#1f6fb2);color:#fff;padding:22px 26px;border-radius:10px">
<h1 style="margin:0;color:#fff">Python for Data Science — Hands-On Workshop</h1>
<h3 style="margin:6px 0 0;color:#dfe9f7">Session 3 of 4 · Data Manipulation with Pandas</h3>
<p style="margin:10px 0 0">📅 <b>Date:</b> 16/11/2026 &nbsp;|&nbsp; 👤 <b>Instructor:</b> Mr. Mahesh Divakaran &nbsp;|&nbsp; 🏷️ <b>Batch:</b> 30</p>
</div>

### 🎯 What you will learn today

| # | Topic | You will be able to… |
|---|-------|----------------------|
| 1 | Series & DataFrame | create labelled 1-D and 2-D data from lists, dicts and NumPy |
| 2 | Inspection | use `head`, `tail`, `info`, `describe`, `value_counts`, `nunique`, memory usage |
| 3 | Reading & writing | save and load CSV files |
| 4 | Selection | pick columns and rows with `[]`, `.loc[]`, `.iloc[]`, set/reset the index |
| 5 | Filtering | keep rows with `&`, `|`, `~`, `isin`, `between`, `query` |
| 6 | New columns | vectorised maths, `.map()`, `.apply()`, `np.where`, `np.select` |
| 7 | Text & dates | `.str` methods, `pd.to_datetime` and the `.dt` accessor |
| 8 | Sorting & ranking | `sort_values`, `sort_index`, `nlargest`, `rank` |

### 🧭 How to use this notebook
- Run each cell with **Shift + Enter**, from top to bottom.
- Read the text cell first, then run the code, then read **"What to observe"**.
- Change the numbers and run again — that is the best way to learn!

> ✏️ Theme: a **clinic visits table** `visits_df` — 25 patients × 4 visits = **100 rows**. Session 4 will clean messy versions of the same patients.

## 0. Importing Pandas

Pandas is built on NumPy. It adds **row labels** and **column names**, so data looks like an Excel sheet.
The usual short names are `pd` and `np`.

In [1]:
import numpy as np
import pandas as pd

print("pandas:", pd.__version__)
print("numpy :", np.__version__)

pandas: 3.0.5
numpy : 2.5.3


> 👀 **What to observe:** Any pandas 2.x or 3.x works. Small display differences between versions (like `object` vs `str` for text columns) are normal.

## 1. Series — One Labelled Column

A **Series** is a NumPy array **plus labels** (the **index**).

| Created from | Example |
|--------------|---------|
| list | `pd.Series([120, 135, 128])` |
| list + index | `pd.Series([120, 135], index=["P001", "P002"])` |
| dictionary | `pd.Series({"P001": 120, "P002": 135})` |
| NumPy array | `pd.Series(np.array([...]))` |

In [2]:
bp = pd.Series([148, 132, 125, 160], index=["P001", "P002", "P003", "P004"], name="systolic_bp")
print(bp)
print("\nValue for P002:", bp["P002"])
print("Mean:", bp.mean())

P001    148
P002    132
P003    125
P004    160
Name: systolic_bp, dtype: int64

Value for P002: 132
Mean: 141.25


In [3]:
# From a dictionary: keys become the index
cholesterol = pd.Series({"P001": 210, "P002": 185, "P003": 240})
print(cholesterol)

# From a NumPy array: default index 0, 1, 2 ...
rng = np.random.default_rng(42)
hr = pd.Series(rng.integers(60, 100, size=4), name="heart_rate")
print("\n", hr, sep="")

P001    210
P002    185
P003    240
dtype: int64

0    63
1    90
2    86
3    77
Name: heart_rate, dtype: int64


> 👀 **What to observe:** The left column is the **index** (labels), the right column is the **values**. `name` labels the Series and becomes the column name inside a DataFrame.

## 2. DataFrame — A Table

A **DataFrame** is a table: many Series that share one index. The easiest way to create one is a **dictionary of lists** — each key becomes a column.

In [4]:
small_df = pd.DataFrame({
    "patient_id": ["P001", "P002", "P003"],
    "age": [45, 32, 67],
    "site": ["Kochi", "Pune", "Delhi"],
})
small_df

,patient_id,age,site
0,P001,45,Kochi
1,P002,32,Pune
2,P003,67,Delhi


In [5]:
# From a NumPy 2-D array (like Session 2) + column names
glucose_table = np.array([[95, 102, 98], [140, 132, 125], [110, 115, 108]])
glucose_df = pd.DataFrame(glucose_table,
                          columns=["visit_1", "visit_2", "visit_3"],
                          index=["P001", "P002", "P003"])
glucose_df

,visit_1,visit_2,visit_3
P001,95,102,98
P002,140,132,125
P003,110,115,108


> 👀 **What to observe:** The NumPy table from Session 2 now has readable row and column labels.

### 2.1 Our Main Dataset: `visits_df`

We simulate a 12-week study: 25 patients, each seen at **Week 0, 4, 8 and 12**.
Patients on **Drug A** show a fall in blood pressure; **Placebo** patients stay about the same.

| Column | Meaning |
|--------|---------|
| `patient_id` | P001 … P025 |
| `site` | clinic city |
| `visit` | Week 0 / 4 / 8 / 12 |
| `visit_date` | date of the visit (stored as **text** for now) |
| `age` | age in years |
| `treatment_arm` | Drug A or Placebo |
| `systolic_bp` | mmHg |
| `cholesterol` | mg/dL |
| `adherence_pct` | % of doses taken |

In [6]:
rng = np.random.default_rng(42)

n_patients = 25
weeks = [0, 4, 8, 12]

# One row per patient: fixed details
patient_ids = [f"P{i:03d}" for i in range(1, n_patients + 1)]
sites = rng.choice(["Kochi", "Pune", "Delhi", "Chennai"], size=n_patients)
ages = rng.integers(28, 76, size=n_patients)
arms = np.where(np.arange(n_patients) % 2 == 0, "Drug A", "Placebo")   # 1:1 allocation
start_dates = pd.Timestamp("2026-01-05") + pd.to_timedelta(rng.integers(0, 30, size=n_patients), unit="D")
base_bp = rng.normal(150, 12, size=n_patients)
base_chol = rng.normal(220, 30, size=n_patients)

# Build the 100 visit rows
rows = []
for i in range(n_patients):
    for week in weeks:
        drop = week * 1.2 if arms[i] == "Drug A" else week * 0.2
        rows.append({
            "patient_id": patient_ids[i],
            "site": sites[i],
            "visit": f"Week {week}",
            "visit_date": (start_dates[i] + pd.Timedelta(weeks=week)).strftime("%Y-%m-%d"),
            "age": int(ages[i]),
            "treatment_arm": arms[i],
            "systolic_bp": int(round(base_bp[i] - drop + rng.normal(0, 4))),
            "cholesterol": int(round(base_chol[i] - week * 0.8 + rng.normal(0, 6))),
            "adherence_pct": round(float(rng.uniform(55, 100)), 1),
        })

visits_df = pd.DataFrame(rows)
print("Rows, columns:", visits_df.shape)
visits_df.head()

Rows, columns: (100, 9)


,patient_id,site,visit,visit_date,age,treatment_arm,systolic_bp,cholesterol,adherence_pct
0,P001,Kochi,Week 0,2026-01-27,58,Drug A,142,240,75.1
1,P001,Kochi,Week 4,2026-02-24,58,Drug A,135,232,83.4
2,P001,Kochi,Week 8,2026-03-24,58,Drug A,124,223,60.3
3,P001,Kochi,Week 12,2026-04-21,58,Drug A,122,230,86.5
4,P002,Chennai,Week 0,2026-01-10,47,Placebo,156,249,90.0


> 👀 **What to observe:** Exactly **100 rows × 9 columns**. Everyone running this gets the same numbers because of `default_rng(42)`.

## 3. Looking at Your Data

Always inspect before you analyse.

| Method | Shows |
|--------|-------|
| `df.head(n)` / `df.tail(n)` | first / last `n` rows (default 5) |
| `df.shape` | (rows, columns) |
| `df.columns` / `df.dtypes` | column names / types |
| `df.info()` | types, non-missing counts, memory |
| `df.describe()` | count, mean, std, min, quartiles, max |
| `df["col"].value_counts()` | how often each value appears |
| `df["col"].nunique()` | number of distinct values |

In [7]:
visits_df.tail(3)

,patient_id,site,visit,visit_date,age,treatment_arm,systolic_bp,cholesterol,adherence_pct
97,P025,Chennai,Week 4,2026-02-22,65,Drug A,156,216,60.5
98,P025,Chennai,Week 8,2026-03-22,65,Drug A,149,211,55.6
99,P025,Chennai,Week 12,2026-04-19,65,Drug A,143,201,68.3


In [8]:
visits_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   patient_id     100 non-null    str    
 1   site           100 non-null    str    
 2   visit          100 non-null    str    
 3   visit_date     100 non-null    str    
 4   age            100 non-null    int64  
 5   treatment_arm  100 non-null    str    
 6   systolic_bp    100 non-null    int64  
 7   cholesterol    100 non-null    int64  
 8   adherence_pct  100 non-null    float64
dtypes: float64(1), int64(3), str(5)
memory usage: 7.2 KB


> 👀 **What to observe:** `info()` shows **100 non-null** in every column (no missing data yet) and the type of each column. `visit_date` is text — we fix that in Section 8.

In [9]:
visits_df.describe().round(1)

,age,systolic_bp,cholesterol,adherence_pct
count,100.0,100.0,100.0,100.0
mean,53.5,146.1,212.0,76.6
std,13.2,12.0,20.6,12.7
min,31.0,116.0,167.0,55.6
25%,45.0,140.8,197.0,66.0
50%,54.0,148.0,211.0,77.4
75%,65.0,153.0,230.2,86.4
max,74.0,175.0,250.0,99.7


> 👀 **What to observe:** `describe()` only summarises **number** columns. Mean systolic BP is around 145 mmHg.

In [10]:
print(visits_df["treatment_arm"].value_counts())
print()
print(visits_df["site"].value_counts(normalize=True).round(2))   # as proportions
print("\nUnique patients:", visits_df["patient_id"].nunique())
print("Unique sites   :", visits_df["site"].nunique())

treatment_arm
Drug A     52
Placebo    48
Name: count, dtype: int64

site
Chennai    0.32
Delhi      0.28
Kochi      0.24
Pune       0.16
Name: proportion, dtype: float64

Unique patients: 25
Unique sites   : 4


> 👀 **What to observe:** Each arm has 52 or 48 rows (13 vs 12 patients × 4 visits). `normalize=True` gives proportions instead of counts. 25 unique patients, as expected.

In [11]:
# Memory usage in bytes per column (deep=True measures text properly)
print(visits_df.memory_usage(deep=True))
print(f"\nTotal: {visits_df.memory_usage(deep=True).sum() / 1024:.1f} KB")

# Repeated text -> 'category' type saves memory
site_as_category = visits_df["site"].astype("category")
print("site as text    :", visits_df["site"].memory_usage(deep=True), "bytes")
print("site as category:", site_as_category.memory_usage(deep=True), "bytes")

Index             132
patient_id       5300
site             7848
visit            5525
visit_date       5900
age               800
treatment_arm    7948
systolic_bp       800
cholesterol       800
adherence_pct     800
dtype: int64

Total: 35.0 KB
site as text    : 7980 bytes
site as category: 545 bytes


> 👀 **What to observe:** Text columns use the most memory. Converting a column with few repeated values (like `site`) to `category` makes it much smaller — very useful for big files.

## 4. Reading and Writing Files

| Write | Read |
|-------|------|
| `df.to_csv("file.csv", index=False)` | `pd.read_csv("file.csv")` |
| `df.to_excel("file.xlsx", index=False)` | `pd.read_excel("file.xlsx")` |

Use `index=False` so the row numbers are not saved as an extra column.
The file is saved in the same folder as this notebook.

In [12]:
visits_df.to_csv("visits_session3.csv", index=False)

loaded_df = pd.read_csv("visits_session3.csv")
print("Loaded shape:", loaded_df.shape)
print("Same data?  :", loaded_df.equals(visits_df))
loaded_df.head(3)

Loaded shape: (100, 9)
Same data?  : True


,patient_id,site,visit,visit_date,age,treatment_arm,systolic_bp,cholesterol,adherence_pct
0,P001,Kochi,Week 0,2026-01-27,58,Drug A,142,240,75.1
1,P001,Kochi,Week 4,2026-02-24,58,Drug A,135,232,83.4
2,P001,Kochi,Week 8,2026-03-24,58,Drug A,124,223,60.3


> 👀 **What to observe:** We saved and loaded 100 rows. `equals()` is `True` — the round trip did not change the data.

In [13]:
# Read only some columns, and parse the date column while reading
subset = pd.read_csv("visits_session3.csv",
                     usecols=["patient_id", "visit_date", "systolic_bp"],
                     parse_dates=["visit_date"])
print(subset.dtypes)

patient_id                str
visit_date     datetime64[us]
systolic_bp             int64
dtype: object


> 👀 **What to observe:** `usecols` loads only what you need; `parse_dates` turns text into real dates (`datetime64`).

## 5. Selecting Columns and Rows

| Code | Gives |
|------|-------|
| `df["age"]` | one column → **Series** |
| `df[["age", "site"]]` | several columns → **DataFrame** (note double brackets) |
| `df.loc[row_label, col_label]` | select by **labels** (end label is **included**) |
| `df.iloc[row_pos, col_pos]` | select by **positions** (end position **excluded**, like lists) |

In [14]:
print(type(visits_df["age"]))
print(type(visits_df[["age", "site"]]))
visits_df[["patient_id", "visit", "systolic_bp"]].head(4)

<class 'pandas.Series'>
<class 'pandas.DataFrame'>


,patient_id,visit,systolic_bp
0,P001,Week 0,142
1,P001,Week 4,135
2,P001,Week 8,124
3,P001,Week 12,122


In [15]:
# loc: labels. Rows 0 to 3 INCLUDED, chosen columns
visits_df.loc[0:3, ["patient_id", "visit", "systolic_bp"]]

,patient_id,visit,systolic_bp
0,P001,Week 0,142
1,P001,Week 4,135
2,P001,Week 8,124
3,P001,Week 12,122


In [16]:
# iloc: positions. Rows 0,1,2 (3 excluded), first 3 columns
visits_df.iloc[0:3, 0:3]

,patient_id,site,visit
0,P001,Kochi,Week 0
1,P001,Kochi,Week 4
2,P001,Kochi,Week 8


> 👀 **What to observe:** `.loc[0:3]` returned **4** rows (label 3 included); `.iloc[0:3]` returned **3** rows (position 3 excluded). This is the most common pandas confusion!

### 5.1 Setting and Resetting the Index

The index can be a meaningful column, such as `patient_id`. Then `.loc` can find rows by patient.
`reset_index()` turns the index back into a normal column.

In [17]:
baseline = visits_df[visits_df["visit"] == "Week 0"].set_index("patient_id")
print(baseline.loc["P005", ["site", "age", "systolic_bp"]])
print()
print(baseline.loc[["P001", "P003"], ["treatment_arm", "systolic_bp"]])

site           Pune
age              49
systolic_bp     143
Name: P005, dtype: object

           treatment_arm  systolic_bp
patient_id                           
P001              Drug A          142
P003              Drug A          157


In [18]:
baseline_reset = baseline.reset_index()
baseline_reset.head(3)

,patient_id,site,visit,visit_date,age,treatment_arm,systolic_bp,cholesterol,adherence_pct
0,P001,Kochi,Week 0,2026-01-27,58,Drug A,142,240,75.1
1,P002,Chennai,Week 0,2026-01-10,47,Placebo,156,249,90.0
2,P003,Delhi,Week 0,2026-01-15,67,Drug A,157,235,89.1


> 👀 **What to observe:** With `patient_id` as the index, `baseline.loc['P005']` finds the patient by name. `reset_index()` brings `patient_id` back as a column.

### 5.2 Changing a Value Safely

Use **one** `.loc[row, column] = value` call. Avoid chained indexing like `df[df.age > 60]["age"] = 0` — it may not change your table (and newer pandas will warn you).

In [19]:
demo = visits_df.head(3).copy()        # .copy() so we don't touch visits_df
demo.loc[demo["patient_id"] == "P001", "site"] = "Kochi (main)"
demo[["patient_id", "site"]]

,patient_id,site
0,P001,Kochi (main)
1,P001,Kochi (main)
2,P001,Kochi (main)


> 👀 **What to observe:** Only P001's rows changed. `.copy()` keeps our main `visits_df` untouched.

## 6. Filtering Rows

A condition gives a True/False Series (a **mask**, like Session 2). Put it inside `df[...]`.

| Tool | Example |
|------|---------|
| single condition | `df[df["age"] > 60]` |
| and / or / not | `df[(cond1) & (cond2)]`, `|`, `~` — **brackets around each condition** |
| `.isin()` | `df[df["site"].isin(["Kochi", "Pune"])]` |
| `.between()` | `df[df["age"].between(40, 60)]` (both ends included) |
| `.query()` | `df.query("age > 60 and site == 'Kochi'")` |

In [20]:
high_bp = visits_df[visits_df["systolic_bp"] >= 160]
print("Visits with SBP >= 160:", len(high_bp))
high_bp[["patient_id", "visit", "systolic_bp"]].head()

Visits with SBP >= 160: 10


,patient_id,visit,systolic_bp
5,P002,Week 4,160
12,P004,Week 0,160
32,P009,Week 0,162
54,P014,Week 8,168
76,P020,Week 0,175


In [21]:
# Drug A AND final visit
drug_a_final = visits_df[(visits_df["treatment_arm"] == "Drug A") & (visits_df["visit"] == "Week 12")]
print("Drug A at Week 12:", len(drug_a_final), "rows")

# Low adherence OR high cholesterol
concern = visits_df[(visits_df["adherence_pct"] < 60) | (visits_df["cholesterol"] > 250)]
print("Low adherence or high cholesterol:", len(concern), "rows")

# NOT Kochi
not_kochi = visits_df[~(visits_df["site"] == "Kochi")]
print("Not Kochi:", len(not_kochi), "rows")

Drug A at Week 12: 13 rows
Low adherence or high cholesterol: 10 rows
Not Kochi: 76 rows


In [22]:
south = visits_df[visits_df["site"].isin(["Kochi", "Chennai"])]
print("Kochi or Chennai rows:", len(south))

middle_age = visits_df[visits_df["age"].between(40, 60)]
print("Age 40-60 rows      :", len(middle_age))

# query(): readable text conditions; use @ for Python variables
bp_limit = 140
visits_df.query("treatment_arm == 'Placebo' and systolic_bp > @bp_limit and visit == 'Week 12'")[
    ["patient_id", "site", "systolic_bp"]
]

Kochi or Chennai rows: 56
Age 40-60 rows      : 44


,patient_id,site,systolic_bp
7,P002,Chennai,148
15,P004,Pune,153
23,P006,Chennai,154
31,P008,Delhi,151
39,P010,Kochi,150
47,P012,Chennai,156
55,P014,Chennai,157
63,P016,Chennai,145
79,P020,Pune,165
87,P022,Pune,151


> 👀 **What to observe:** Each filter returns only matching rows. `query()` reads almost like English; `@bp_limit` uses a normal Python variable inside the text.

## 7. Creating New Columns

| Method | Use when | Speed |
|--------|----------|-------|
| column maths `df["a"] * 2` | simple formulas | ⚡ fastest |
| `.map(dict)` | replace codes with labels | ⚡ fast |
| `np.where(cond, a, b)` | two choices | ⚡ fast |
| `np.select([conds], [choices], default)` | many choices | ⚡ fast |
| `.apply(function)` | complex custom logic | 🐢 slow (row by row) |

Rule of thumb: try vectorised methods first; use `.apply()` only when nothing else works.

In [23]:
df = visits_df.copy()            # work on a copy called df from now on

# Vectorised maths
df["chol_mmol"] = (df["cholesterol"] / 38.67).round(2)
df["missed_pct"] = 100 - df["adherence_pct"]

# .map() with a dictionary: city -> region
region_map = {"Kochi": "South", "Chennai": "South", "Pune": "West", "Delhi": "North"}
df["region"] = df["site"].map(region_map)

df[["patient_id", "site", "region", "cholesterol", "chol_mmol", "missed_pct"]].head()

,patient_id,site,region,cholesterol,chol_mmol,missed_pct
0,P001,Kochi,South,240,6.21,24.9
1,P001,Kochi,South,232,6.00,16.6
2,P001,Kochi,South,223,5.77,39.7
3,P001,Kochi,South,230,5.95,13.5
4,P002,Chennai,South,249,6.44,10.0


In [24]:
# np.where: two categories
df["adherence_flag"] = np.where(df["adherence_pct"] >= 80, "Good", "Poor")

# np.select: several categories (checked in order, first match wins)
conditions = [
    df["systolic_bp"] >= 140,
    df["systolic_bp"] >= 130,
    df["systolic_bp"] >= 120,
]
labels = ["Stage 2", "Stage 1", "Elevated"]
df["bp_category"] = np.select(conditions, labels, default="Normal")

print(df["adherence_flag"].value_counts())
print()
print(df["bp_category"].value_counts())

adherence_flag
Poor    56
Good    44
Name: count, dtype: int64

bp_category
Stage 2     76
Stage 1     11
Elevated    11
Normal       2
Name: count, dtype: int64


In [25]:
# .apply(): custom function, row by row (axis=1)
def risk_score(row):
    score = 0
    if row["age"] >= 60:
        score += 1
    if row["systolic_bp"] >= 140:
        score += 1
    if row["cholesterol"] >= 240:
        score += 1
    return score

df["risk_score"] = df.apply(risk_score, axis=1)

# .apply() on one column with a lambda
df["age_group"] = df["age"].apply(lambda a: "60+" if a >= 60 else "<60")

df[["patient_id", "age", "systolic_bp", "cholesterol", "risk_score", "age_group"]].head()

,patient_id,age,systolic_bp,cholesterol,risk_score,age_group
0,P001,58,142,240,2,<60
1,P001,58,135,232,0,<60
2,P001,58,124,223,0,<60
3,P001,58,122,230,0,<60
4,P002,47,156,249,2,<60


> 👀 **What to observe:** `np.select` is like `if/elif/else` for a whole column — the same idea as Session 1, but without a loop. `.apply(axis=1)` passes each row to your function.

## 8. Text (`.str`) and Dates (`.dt`)

**`.str`** applies string methods to a whole column: `.str.upper()`, `.str.lower()`, `.str.strip()`, `.str.contains()`, `.str.replace()`, `.str[0:3]`.

**Dates:** `pd.to_datetime()` turns text into real dates. Then **`.dt`** gives parts: `.dt.year`, `.dt.month`, `.dt.day_name()` …
Subtracting two dates gives a duration; `.dt.days` turns it into a number.

In [26]:
df["site_code"] = df["site"].str.upper().str[:3]           # KOCHI -> KOC
df["patient_num"] = df["patient_id"].str.replace("P", "").astype(int)
df["is_week0"] = df["visit"].str.contains("Week 0")
df["week_num"] = df["visit"].str.extract(r"(\d+)", expand=False).astype(int)   # number from text

df[["patient_id", "patient_num", "site", "site_code", "visit", "week_num", "is_week0"]].head(5)

,patient_id,patient_num,site,site_code,visit,week_num,is_week0
0,P001,1,Kochi,KOC,Week 0,0,True
1,P001,1,Kochi,KOC,Week 4,4,False
2,P001,1,Kochi,KOC,Week 8,8,False
3,P001,1,Kochi,KOC,Week 12,12,False
4,P002,2,Chennai,CHE,Week 0,0,True


> 👀 **What to observe:** `.str.extract(r'(\d+)')` pulls the number out of `'Week 12'` → `12`. All of this works on 100 rows without a loop.

In [27]:
print("Before:", df["visit_date"].dtype)
df["visit_date"] = pd.to_datetime(df["visit_date"])
print("After :", df["visit_date"].dtype)

df["visit_year"] = df["visit_date"].dt.year
df["visit_month"] = df["visit_date"].dt.month_name()
df["visit_weekday"] = df["visit_date"].dt.day_name()

# Days since each patient's first visit
first_visit = df.groupby("patient_id")["visit_date"].transform("min")   # preview of Session 4
df["study_day"] = (df["visit_date"] - first_visit).dt.days

df[["patient_id", "visit", "visit_date", "visit_month", "visit_weekday", "study_day"]].head(8)

Before: str
After : datetime64[us]


,patient_id,visit,visit_date,visit_month,visit_weekday,study_day
0,P001,Week 0,2026-01-27,January,Tuesday,0
1,P001,Week 4,2026-02-24,February,Tuesday,28
2,P001,Week 8,2026-03-24,March,Tuesday,56
3,P001,Week 12,2026-04-21,April,Tuesday,84
4,P002,Week 0,2026-01-10,January,Saturday,0
5,P002,Week 4,2026-02-07,February,Saturday,28
6,P002,Week 8,2026-03-07,March,Saturday,56
7,P002,Week 12,2026-04-04,April,Saturday,84


> 👀 **What to observe:** After `to_datetime`, the column type is `datetime64`. `study_day` is 0, 28, 56, 84 — exactly 4 weeks apart.

## 9. Sorting and Ranking

| Method | Does |
|--------|------|
| `sort_values("col")` | sort rows by a column (`ascending=False` for highest first) |
| `sort_values(["a", "b"])` | sort by several columns |
| `sort_index()` | sort by the index |
| `nlargest(n, "col")` / `nsmallest(n, "col")` | top / bottom `n` rows |
| `rank()` | position of each value (1 = smallest by default) |

In [28]:
final = df[df["visit"] == "Week 12"]

final.sort_values("systolic_bp", ascending=False)[["patient_id", "treatment_arm", "systolic_bp"]].head(5)

,patient_id,treatment_arm,systolic_bp
79,P020,Placebo,165
55,P014,Placebo,157
47,P012,Placebo,156
23,P006,Placebo,154
15,P004,Placebo,153


In [29]:
# Sort by arm (A-Z), then BP (highest first)
final.sort_values(["treatment_arm", "systolic_bp"], ascending=[True, False])[
    ["patient_id", "treatment_arm", "systolic_bp"]
].head(6)

,patient_id,treatment_arm,systolic_bp
35,P009,Drug A,152
43,P011,Drug A,148
11,P003,Drug A,145
99,P025,Drug A,143
75,P019,Drug A,139
27,P007,Drug A,137


In [30]:
print("Top 3 cholesterol at Week 12:")
print(final.nlargest(3, "cholesterol")[["patient_id", "cholesterol"]])
print("\nLowest 3 adherence at Week 12:")
print(final.nsmallest(3, "adherence_pct")[["patient_id", "adherence_pct"]])

Top 3 cholesterol at Week 12:
   patient_id  cholesterol
11       P003          236
7        P002          234
23       P006          233

Lowest 3 adherence at Week 12:
   patient_id  adherence_pct
19       P005           56.0
79       P020           59.3
87       P022           60.4


In [31]:
ranked = final[["patient_id", "systolic_bp"]].copy()
ranked["bp_rank"] = ranked["systolic_bp"].rank(ascending=False, method="min").astype(int)   # 1 = highest
ranked = ranked.set_index("patient_id").sort_index()   # sort by patient ID
ranked.head(6)

,systolic_bp,bp_rank
patient_id,,
P001,122,22
P002,148,10
P003,145,13
P004,153,5
P005,126,21
P006,154,4


> 👀 **What to observe:** `rank(ascending=False)` gives **1** to the highest BP. `method='min'` gives tied values the same (lowest) rank. `sort_index()` puts patients back in ID order.

## 10. Putting It Together — Week 12 Snapshot

Filter → new column → sort → select. A quick preview of `groupby` (Session 4) shows the treatment effect.

In [32]:
week0 = df[df["visit"] == "Week 0"].set_index("patient_id")["systolic_bp"]
week12 = df[df["visit"] == "Week 12"].set_index("patient_id")

snapshot = week12[["site", "treatment_arm", "systolic_bp"]].copy()
snapshot["bp_change"] = snapshot["systolic_bp"] - week0        # pandas lines up rows by patient_id
snapshot = snapshot.sort_values("bp_change")

print("Mean BP change by arm (mmHg):")
print(snapshot.groupby("treatment_arm")["bp_change"].mean().round(1))
snapshot.head()

Mean BP change by arm (mmHg):
treatment_arm
Drug A    -13.9
Placebo    -2.7
Name: bp_change, dtype: float64


,site,treatment_arm,systolic_bp,bp_change
patient_id,,,,
P021,Delhi,Drug A,121,-26
P001,Kochi,Drug A,122,-20
P017,Delhi,Drug A,128,-17
P005,Pune,Drug A,126,-17
P015,Delhi,Drug A,116,-16


> 👀 **What to observe:** Drug A patients show a fall of about **14 mmHg**; Placebo changes only a little. When you subtract two Series, pandas matches rows by their **index labels** automatically.

---
## 🧪 Practice Exercises
Use `visits_df` (the original) for all exercises.

### Exercise 1 — Beginner: Explore
1. Print the shape and the column names.
2. How many visits happened at each **site**? (`value_counts`)
3. Show the `patient_id`, `visit` and `cholesterol` of the **first 5 rows** using `.loc`, then using `.iloc`.

In [33]:
# Exercise 1 - write your code here

### Exercise 2 — Intermediate: Filter and flag
1. Keep only **Week 12** visits for patients aged **50 to 70** in **Pune or Delhi**.
2. Add a column `chol_flag` = `"High"` if cholesterol ≥ 240, `"Borderline"` if ≥ 200, else `"Desirable"` (use `np.select`).
3. Sort the result by cholesterol, highest first.

In [34]:
# Exercise 2 - write your code here

### Exercise 3 — Challenge: Visit calendar report
1. Make a copy of `visits_df`, convert `visit_date` to datetime.
2. Add `visit_month` (e.g. `"January"`) and `weekday`.
3. How many visits fall on a **Saturday or Sunday**? List those patient IDs (unique).
4. For **Week 12** only, rank patients by `adherence_pct` (1 = best) and show the top 5 with `nsmallest` on the rank.
5. Save the Week 12 table to `week12_report.csv` (no index) and read it back to check the row count.

In [35]:
# Exercise 3 - write your code here

---
## ✅ Solutions
The solution cells are **collapsed** — open them after you try.

In [36]:
#@title Solution 1 — Explore
print("Shape:", visits_df.shape)
print("Columns:", list(visits_df.columns))
print()
print(visits_df["site"].value_counts())
print()
print(visits_df.loc[0:4, ["patient_id", "visit", "cholesterol"]])
print()
print(visits_df.iloc[0:5, [0, 2, 7]])        # positions of the same 3 columns

Shape: (100, 9)
Columns: ['patient_id', 'site', 'visit', 'visit_date', 'age', 'treatment_arm', 'systolic_bp', 'cholesterol', 'adherence_pct']

site
Chennai    32
Delhi      28
Kochi      24
Pune       16
Name: count, dtype: int64

  patient_id    visit  cholesterol
0       P001   Week 0          240
1       P001   Week 4          232
2       P001   Week 8          223
3       P001  Week 12          230
4       P002   Week 0          249

  patient_id    visit  cholesterol
0       P001   Week 0          240
1       P001   Week 4          232
2       P001   Week 8          223
3       P001  Week 12          230
4       P002   Week 0          249


In [37]:
#@title Solution 2 — Filter and flag
ex2 = visits_df[
    (visits_df["visit"] == "Week 12")
    & (visits_df["age"].between(50, 70))
    & (visits_df["site"].isin(["Pune", "Delhi"]))
].copy()

ex2["chol_flag"] = np.select(
    [ex2["cholesterol"] >= 240, ex2["cholesterol"] >= 200],
    ["High", "Borderline"],
    default="Desirable",
)

ex2.sort_values("cholesterol", ascending=False)[["patient_id", "site", "age", "cholesterol", "chol_flag"]]

,patient_id,site,age,cholesterol,chol_flag
11,P003,Delhi,67,236,Borderline
51,P013,Delhi,67,226,Borderline
59,P015,Delhi,58,218,Borderline
67,P017,Delhi,64,203,Borderline
15,P004,Pune,54,191,Desirable


In [38]:
#@title Solution 3 — Visit calendar report
ex3 = visits_df.copy()
ex3["visit_date"] = pd.to_datetime(ex3["visit_date"])
ex3["visit_month"] = ex3["visit_date"].dt.month_name()
ex3["weekday"] = ex3["visit_date"].dt.day_name()

weekend = ex3[ex3["weekday"].isin(["Saturday", "Sunday"])]
print("Weekend visits:", len(weekend))
print("Patients:", sorted(weekend["patient_id"].unique()))

wk12 = ex3[ex3["visit"] == "Week 12"].copy()
wk12["adherence_rank"] = wk12["adherence_pct"].rank(ascending=False, method="min").astype(int)
print("\nTop 5 adherence at Week 12:")
print(wk12.nsmallest(5, "adherence_rank")[["patient_id", "adherence_pct", "adherence_rank"]])

wk12.to_csv("week12_report.csv", index=False)
print("\nRows read back:", len(pd.read_csv("week12_report.csv")))

Weekend visits: 32
Patients: ['P002', 'P010', 'P011', 'P015', 'P017', 'P022', 'P023', 'P025']

Top 5 adherence at Week 12:
   patient_id  adherence_pct  adherence_rank
95       P024           99.4               1
43       P011           98.0               2
47       P012           91.2               3
3        P001           86.5               4
59       P015           85.6               5

Rows read back: 25


---
### 📌 Session 3 Summary

| Topic | Remember |
|-------|----------|
| Series / DataFrame | labelled column / labelled table |
| Inspect | `head`, `info`, `describe`, `value_counts`, `nunique`, `memory_usage(deep=True)` |
| Files | `to_csv(index=False)`, `read_csv(usecols=..., parse_dates=...)` |
| Select | `df["col"]`, `df[["a","b"]]`, `.loc` (labels, end included), `.iloc` (positions, end excluded) |
| Filter | brackets + `&` `|` `~`, `isin`, `between`, `query` |
| New columns | maths → `.map` → `np.where` / `np.select` → `.apply` last |
| Text & dates | `.str.upper/strip/extract`, `pd.to_datetime`, `.dt.year/day_name/days` |
| Sort | `sort_values`, `sort_index`, `nlargest`, `rank` |

**Next — Session 4: Data Wrangling**, where we clean messy versions of these same patients, merge tables and reshape them. 🚀